In [2]:
!pip install catboost

     |████████████████████████████████| 76.2 MB 34.0 MB/s            


In [19]:
## import the libraries needed
import pandas as pd
import numpy as np


# Here we import our dataset from the CatBoost dataset library
from catboost.datasets import titanic


In [20]:
## The titanic dataset is made up of the train and test set, so we have to separate the data
titanic_train, titanic_test = titanic()

## Here we create a list to sort the columns so that the "Survived" column comes last
## This is because "Survived" is the target
column_sort = [ 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket',
'Fare', 'Cabin', 'Embarked','Survived']

## Now we apply the sorted columns to the train data
train = titanic_train[column_sort]
train.set_index('Pclass') ## Not necessary just to get of the default index

test = titanic_test
train.head()

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,0
1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1
4,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,0


In [12]:
## Remember the target column - "Survived" - we identified in the cell above,
## it is missing in the test set
## To solve this problem, we would create a column and fill it with dummy values,
## let's say '2' so it is not dormant and we can merge the DataFrame later
test['Survived'] = 2  ## The numpy background of pandas allows this to work
test.sample(5) ## shows five random rows in the dataset

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Survived
254,1146,3,"Wenzel, Mr. Linhart",male,32.5,0,0,345775,9.5000,NaN,S,2
208,1100,1,"Rosenbaum, Miss. Edith Louise",female,33.0,0,0,PC 17613,27.7208,A11,C,2
14,906,1,"Chaffee, Mrs. Herbert Fuller (Carrie Constance...",female,47.0,1,0,W.E.P. 5734,61.1750,E31,S,2
91,983,3,"Pedersen, Mr. Olaf",male,NaN,0,0,345498,7.7750,NaN,S,2
293,1185,1,"Dodge, Dr. Washington",male,53.0,1,1,33638,81.8583,A34,S,2


In [21]:
## We would combine the train and test set into one DataFrame,
## so we do not have to repeat the same process for the two sets
df = pd.concat([train,test],ignore_index = False)

## Some features (such as Name, and Age) are irrelevant so we delete them
df = df.drop(['Name', 'Age'], axis=1)

## The data is not clean so we check all the columns for missing values
df.isnull().sum(axis=0)

Pclass            0
Sex               0
SibSp             0
Parch             0
Ticket            0
Fare              1
Cabin          1014
Embarked          2
Survived        418
PassengerId     891
dtype: int64

In [22]:
## "Fare", "Cabin", "Embarked", and "PassengerId" have missing values, we have to fix this
df['Embarked'] = df['Embarked'].fillna('S') ## The missing values in Embarked is filled with "S" (for Southampton), the most common value observed in that column
df['Cabin'] = df['Cabin'].fillna('Undefined')
df.fillna(-999, inplace=True)

## Now that the data looks good, we have to separate the train from the test set
train = df[df.Survived != 2]

test = df[df.Survived == 2]
test = test.drop(['Survived'], axis=1) ## drop the placeholder we created earlier in the test set

## Pop out the training features from the target variable
target = train.pop('Survived')
target.head()

0    0.0
1    1.0
2    1.0
3    1.0
4    0.0
Name: Survived, dtype: float64

In [23]:
## Let's ensure the model is trained and fit well
cat_features_index = np.where(train.dtypes != float)[0]

## Split the data into a train and test set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(train, target,
train_size=0.85, random_state=1234)

In [27]:
cat_features_index
train.head()

,Pclass,Sex,SibSp,Parch,Ticket,Fare,Cabin,Embarked,PassengerId
0,3,male,1,0,A/5 21171,7.2500,Undefined,S,-999.0
1,1,female,1,0,PC 17599,71.2833,C85,C,-999.0
2,3,female,0,0,STON/O2. 3101282,7.9250,Undefined,S,-999.0
3,1,female,1,0,113803,53.1000,C123,S,-999.0
4,3,male,0,0,373450,8.0500,Undefined,S,-999.0


In [9]:
## Import the CatBoostClassifier to fit the model and run a prediction
from catboost import CatBoostClassifier
model = CatBoostClassifier(
    custom_loss=['Accuracy'],
    random_seed=42)

## Set the metric for evaluation
model = CatBoostClassifier(eval_metric='Accuracy',
use_best_model=True,  random_seed=42)

model.fit(X_train, y_train, cat_features=cat_features_index,
eval_set=(X_test, y_test))


from catboost import cv
from sklearn.metrics import accuracy_score

print('the test accuracy is :{:.6f}'.format(accuracy_score(
y_test, model.predict(X_test))))

Learning rate set to 0.029583
0:	learn: 0.8031704	test: 0.8059701	best: 0.8059701 (0)	total: 53.1ms	remaining: 53s
1:	learn: 0.8137384	test: 0.8059701	best: 0.8059701 (0)	total: 56.8ms	remaining: 28.3s
2:	learn: 0.8150594	test: 0.8059701	best: 0.8059701 (0)	total: 58.2ms	remaining: 19.3s
3:	learn: 0.8177015	test: 0.8059701	best: 0.8059701 (0)	total: 61ms	remaining: 15.2s
4:	learn: 0.8163804	test: 0.8059701	best: 0.8059701 (0)	total: 63.3ms	remaining: 12.6s
5:	learn: 0.8163804	test: 0.8059701	best: 0.8059701 (0)	total: 64.2ms	remaining: 10.6s
6:	learn: 0.8150594	test: 0.8059701	best: 0.8059701 (0)	total: 66.4ms	remaining: 9.42s
7:	learn: 0.8163804	test: 0.8059701	best: 0.8059701 (0)	total: 67.5ms	remaining: 8.37s
8:	learn: 0.8110964	test: 0.8059701	best: 0.8059701 (0)	total: 71.3ms	remaining: 7.86s
9:	learn: 0.8137384	test: 0.8059701	best: 0.8059701 (0)	total: 73.5ms	remaining: 7.28s
10:	learn: 0.8150594	test: 0.8059701	best: 0.8059701 (0)	total: 75.5ms	remaining: 6.79s
11:	learn: 0.816

In [28]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(max_depth=2, random_state=0)